# Qwen 3.5 27B - Colab Host with vLLM + Tool Calling\n\nLoads Qwen 3.5 27B (Q4_K_M GGUF) via **vLLM** on a Colab A100 and exposes an OpenAI-compatible API with native tool-calling support.\n\n**Why vLLM?** vLLM's built-in `--tool-call-parser qwen` converts Qwen's XML-style tool output into proper OpenAI `tool_calls` objects, so Pi (and other agents) can actually execute tools.

## 1. Check GPU

In [ ]:
#@title 1. Check GPU\nimport subprocess\n\nresult = subprocess.run(\n    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],\n    capture_output=True, text=True\n)\nif result.returncode != 0:\n    print('No GPU found. Enable GPU in Runtime > Change runtime type.')\nelse:\n    print(result.stdout.strip())\n    free_mb = int(result.stdout.strip().split(',')[2].strip().split()[0])\n    if free_mb < 18000:\n        print(f'WARNING: Only {free_mb}MB free. Model needs ~18GB. Consider resetting runtime.')\n    else:\n        print(f'GPU has {free_mb}MB free - should be sufficient.')

## 2. Install vLLM + HuggingFace Hub\n\nThis installs vLLM with CUDA support. On Colab this usually finds a pre-built wheel; if not, it compiles from source (~10-15 min).

In [ ]:
#@title 2. Install vLLM\nimport os, sys\n\nprint('Installing vLLM...')\nprint('This may take 5-15 minutes if building from source.')\n\n# Try standard install first\nos.system('pip install --upgrade vllm huggingface-hub')\n\n# Verify\ntry:\n    import vllm\n    print(f'vLLM version: {vllm.__version__}')\nexcept ImportError as e:\n    print(f'ERROR: {e}')

## 3. Download Model Weights + Tokenizer\n\nvLLM needs the GGUF file **and** the tokenizer config (so it knows the chat template for tool calling).\n\n- **GGUF**: `unsloth/Qwen3.5-27B-GGUF` — the quantized weights\n- **Tokenizer**: `Qwen/Qwen3.5-32B` — config files for chat template & tool parser

In [ ]:
#@title 3. Download Model\nfrom huggingface_hub import hf_hub_download, snapshot_download\n\nGGUF_REPO = "unsloth/Qwen3.5-27B-GGUF"\nGGUF_FILE = "Qwen3.5-27B-Q4_K_M.gguf"\nBASE_MODEL = "Qwen/Qwen3.5-32B"  # for tokenizer/chat template\n\n# Download GGUF weights\nmodel_path = hf_hub_download(\n    repo_id=GGUF_REPO,\n    filename=GGUF_FILE,\n    local_dir="/content/model",\n)\nprint(f'GGUF downloaded to: {model_path}')\n\n# Download tokenizer + config from base model (needed for tool calling)\nprint(f'Downloading tokenizer from {BASE_MODEL}...')\nsnapshot_download(\n    repo_id=BASE_MODEL,\n    local_dir="/content/model_hf",\n    allow_patterns=["tokenizer*.json", "config.json"],\n)\nprint('Tokenizer config ready.')

## 4. Start vLLM OpenAI Server\n\nKey flags:\n- `--quantization gguf` — load the GGUF file\n- `--tokenizer /content/model_hf` — use the base model's tokenizer & chat template\n- `--enable-auto-tool-choice` — allow the model to choose when to call tools\n- `--tool-call-parser qwen` — parse Qwen's XML tool format into OpenAI `tool_calls`\n- `--max-model-len 32768` — set context size (increase to 65536 if you have headroom)

In [ ]:
#@title 4. Start vLLM Server (OpenAI-compatible)\nimport subprocess, time, urllib.request, json\n\nPORT = 8000\nMAX_MODEL_LEN = 32768  #@param {type:"integer"}\n\n# Kill any previous server on the same port\n!lsof -ti :{PORT} | xargs kill -9 2>/dev/null || true\n\n# Open log files\nstdout_log = open("/content/server_stdout.log", "w")\nstderr_log = open("/content/server_stderr.log", "w")\n\nserver_proc = subprocess.Popen(\n    [\n        "python", "-m", "vllm.entrypoints.openai.api_server",\n        "--model", "/content/model/Qwen3.5-27B-Q4_K_M.gguf",\n        "--tokenizer", "/content/model_hf",\n        "--quantization", "gguf",\n        "--dtype", "auto",\n        "--max-model-len", str(MAX_MODEL_LEN),\n        "--gpu-memory-utilization", "0.95",\n        "--enable-auto-tool-choice",\n        "--tool-call-parser", "qwen",\n        "--port", str(PORT),\n        "--host", "0.0.0.0",\n    ],\n    stdout=stdout_log,\n    stderr=stderr_log,\n)\n\nprint(f'Server starting on port {PORT} (PID: {server_proc.pid})...')\nprint('Waiting for model to load (1-3 minutes on A100)...')\n\nfor i in range(180):  # up to 15 minutes\n    time.sleep(5)\n    if server_proc.poll() is not None:\n        print(f'\nServer process exited with code {server_proc.returncode}!')\n        print('Last 30 lines of stderr:')\n        !tail -30 /content/server_stderr.log 2>/dev/null\n        break\n    if i % 6 == 5:\n        print(f'  Still loading... ({(i+1)*5}s)', flush=True)\n    try:\n        req = urllib.request.Request(f'http://localhost:{PORT}/v1/models', method='GET')\n        resp = urllib.request.urlopen(req, timeout=5)\n        if resp.status == 200:\n            data = json.loads(resp.read())\n            model_id = data['data'][0]['id'] if data.get('data') else 'unknown'\n            print(f'\nServer is healthy! Model: {model_id}')\n            break\n    except Exception:\n        pass\nelse:\n    print('\nServer not ready after 15 minutes.')\n    !tail -30 /content/server_stderr.log 2>/dev/null

## 5. Install & Start Cloudflare Tunnel\n\nSame tunnel setup as before. The tunnel URL changes every runtime restart.

In [ ]:
#@title 5. Install & Start Cloudflare Tunnel\nimport subprocess, time\n\n!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared\n!chmod +x /usr/local/bin/cloudflared\n\nPORT = 8000\ntunnel_proc = subprocess.Popen(\n    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],\n    stdout=subprocess.PIPE,\n    stderr=subprocess.PIPE,\n)\n\nprint('Starting tunnel...')\n\nTUNNEL_URL = None\ntimeout_secs = 30\nstart = time.time()\n\nwhile time.time() - start < timeout_secs:\n    line = tunnel_proc.stderr.readline().decode(errors='replace')\n    if not line and tunnel_proc.poll() is not None:\n        print(f'Tunnel process exited unexpectedly with code {tunnel_proc.returncode}')\n        break\n    if 'https://' in line and '.trycloudflare.com' in line:\n        for part in line.split():\n            if part.startswith('https://') and '.trycloudflare.com' in part:\n                TUNNEL_URL = part.rstrip('/')\n                break\n    if TUNNEL_URL:\n        break\n\nif TUNNEL_URL:\n    print(f'\n{"="*60}')\n    print(f'YOUR TUNNEL URL: {TUNNEL_URL}')\n    print(f'{"="*60}')\n    print(f'\nUse it as:')\n    print(f"  base_url = '{TUNNEL_URL}/v1'")\n    print(f'\nRun locally:')\n    print(f"  update-tunnel {TUNNEL_URL}/v1")\nelse:\n    print('\nTunnel URL not found within 30s.')

## 6. Quick Test (with tool calling)

In [ ]:
#@title 6. Test Tool Calling\nimport json, urllib.request\n\nPORT = 8000\n\npayload = {\n    "model": "default",\n    "messages": [{"role": "user", "content": "List files in /content using a tool"}],\n    "tools": [\n        {\n            "type": "function",\n            "function": {\n                "name": "bash",\n                "description": "Run a shell command",\n                "parameters": {\n                    "type": "object",\n                    "properties": {\n                        "command": {"type": "string"}\n                    },\n                    "required": ["command"]\n                }\n            }\n        }\n    ],\n    "stream": False\n}\n\nreq = urllib.request.Request(\n    f'http://localhost:{PORT}/v1/chat/completions',\n    data=json.dumps(payload).encode(),\n    headers={'Content-Type': 'application/json'},\n    method='POST'\n)\n\ntry:\n    resp = urllib.request.urlopen(req, timeout=60)\n    data = json.loads(resp.read())\n    msg = data['choices'][0]['message']\n    print('=== Response ===')\n    print(json.dumps(msg, indent=2))\n    print()\n    if msg.get('tool_calls'):\n        print(f'✅ Tool calls detected: {len(msg["tool_calls"])} call(s)')\n        for tc in msg['tool_calls']:\n            print(f'   Function: {tc["function"]["name"]}')\n            print(f'   Args: {tc["function"]["arguments"]}')\n    else:\n        print('⚠️  No tool_calls in response. Content:')\n        print(msg.get('content', '(empty)'))\nexcept Exception as e:\n    print(f'ERROR: {e}')

## 7. Debug (run this if something goes wrong)

In [ ]:
#@title 7. Debug\nimport subprocess\n\nprint('=== Server Process ===')\nif 'server_proc' in globals():\n    print(f'PID: {server_proc.pid}, Running: {server_proc.poll() is None}')\n    if server_proc.poll() is not None:\n        print(f'Exit code: {server_proc.returncode}')\nelse:\n    print('No server_proc variable found. Has Cell 4 been run?')\n\nprint('\n=== Last 40 lines of server stderr ===')\n!tail -40 /content/server_stderr.log 2>/dev/null || echo 'No log file found'\n\nprint('\n=== Last 20 lines of server stdout ===')\n!tail -20 /content/server_stdout.log 2>/dev/null || echo 'No log file found'\n\nprint('\n=== GPU Memory ===')\n!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader\n\nprint('\n=== Port 8000 Check ===')\n!lsof -i :8000 2>/dev/null || echo 'Nothing listening on port 8000'\n\nprint('\n=== Tunnel Process ===')\nif 'tunnel_proc' in globals():\n    print(f'PID: {tunnel_proc.pid}, Running: {tunnel_proc.poll() is None}')\nelse:\n    print('No tunnel_proc variable found. Has Cell 5 been run?')\n\nif 'TUNNEL_URL' in globals() and TUNNEL_URL:\n    print(f'\nTunnel URL: {TUNNEL_URL}')\nelse:\n    print('\nNo tunnel URL found.')

## Local Usage\n\nAfter running all cells, copy the tunnel URL from Cell 5 output.\n\n**Update Pi config:**\n```bash\nupdate-tunnel https://xxx-yyy-zzz.trycloudflare.com/v1\n```\n\n**Then launch Pi:**\n```bash\npi --provider colab-qwen --model default\n```